# Notebook 6 - Retrieval Metrics (Classical IR)
### Rancang Bangun Chatbot Sumber Informasi Sivitas UPI Berbasis RAG

Independent evaluation alongside RAGAS using **classical information-retrieval metrics**.
RAGAS scores generation quality with an LLM judge; this notebook scores **retrieval quality**
with deterministic math, making it transparent and reproducible for a thesis defense.

| Metric | What it measures |
|---|---|
| **Hit Rate @ k** | Fraction of queries where at least one relevant chunk appears in top-k |
| **Precision @ k** | Of the k retrieved chunks, how many are relevant |
| **Recall @ k** | Of all relevant chunks, how many appear in top-k |
| **MRR (Mean Reciprocal Rank)** | 1 / rank of the first relevant hit, averaged over queries |
| **nDCG @ k** | Discounted gain — relevance weighted by position |
| **Latency** | Embedding + FAISS search time, ms |

**Ground truth** is derived from the `context` column in `evaluation.csv` via substring matching against retrieved chunks. This is a common offline-eval setup when full gold labels aren't available.

## Cell 1 - Imports + load FAISS

In [ ]:
import os, json, time, math, re
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

DRIVE_BASE = Path(os.environ.get("RAG_UPI_BASE", r"D:\Project\RAG_UPI\Dataset"))
PIPE = DRIVE_BASE / "_pipeline"
INDEX_FILE = PIPE / "index" / "faiss.index"
META_FILE  = PIPE / "index" / "chunks_meta.json"
EVAL_DIR   = PIPE / "eval"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "embedding_model": "intfloat/multilingual-e5-base",
    "use_e5_prefixes": True,
    "top_ks": [1, 3, 5, 10],     # we report each metric at multiple cut-offs
    "match_overlap": 0.30,        # gold chunk is "relevant" if 30% of its words appear in retrieved chunk
    "max_eval_questions": 50,
    "csv_path": DRIVE_BASE / "evaluation.csv",
}

index = faiss.read_index(str(INDEX_FILE))
meta  = json.loads(META_FILE.read_text(encoding="utf-8"))
embedder = SentenceTransformer(CONFIG["embedding_model"])

def embed(texts, kind="passage"):
    if CONFIG["use_e5_prefixes"]:
        tag = "query: " if kind == "query" else "passage: "
        texts = [tag + t for t in texts]
    return embedder.encode(texts, convert_to_numpy=True,
                           normalize_embeddings=True).astype("float32")

def retrieve(query, k):
    qv = embed([query], kind="query")
    scores, idxs = index.search(qv, k)
    return [{**meta[i], "score": float(s)} for s, i in zip(scores[0], idxs[0]) if i >= 0]

print(f"FAISS: {index.ntotal} vectors, {index.d} dims")
print(f"Meta: {len(meta)} chunks")


## Cell 2 - Load evaluation queries + define relevance

In [ ]:
df = pd.read_csv(CONFIG["csv_path"]).dropna(subset=["query"]).head(CONFIG["max_eval_questions"])
print(f"Loaded {len(df)} eval queries from {CONFIG['csv_path']}")

_word_re = re.compile(r"[A-Za-zÀ-ÿ]{4,}")

def tokens(s):
    return {w.lower() for w in _word_re.findall(s or "")}

def is_relevant(gold_context: str, retrieved_text: str, overlap_threshold: float):
    """A retrieved chunk is relevant if it shares >=`threshold` of the gold context's content words."""
    gold_tok = tokens(gold_context)
    if len(gold_tok) < 5:
        # Fallback to a fuzzy substring match if gold is too short to tokenise.
        return (gold_context or "")[:80].lower() in (retrieved_text or "").lower()
    ret_tok = tokens(retrieved_text)
    if not ret_tok: return False
    overlap = len(gold_tok & ret_tok) / max(len(gold_tok), 1)
    return overlap >= overlap_threshold


## Cell 3 - Compute per-query retrieval metrics

In [ ]:
def dcg(rels):
    """Standard DCG: sum(rel_i / log2(i+2)) for i in 0..n-1."""
    return sum(r / math.log2(i + 2) for i, r in enumerate(rels))

def evaluate_query(query, gold_context, k_values, overlap):
    """Return a dict of metrics at each k for this single query."""
    max_k = max(k_values)
    t0 = time.time()
    hits = retrieve(query, max_k)
    latency_ms = (time.time() - t0) * 1000

    rels = [1 if is_relevant(gold_context, h["text"], overlap) else 0 for h in hits]

    # First relevant rank for MRR
    first = next((i + 1 for i, r in enumerate(rels) if r), 0)
    rr = (1.0 / first) if first else 0.0

    # Ideal DCG: relevant docs concentrated at the top
    ideal_rels = sorted(rels, reverse=True)

    row = {"query": query, "latency_ms": round(latency_ms, 1),
           "n_relevant_in_topK": sum(rels), "mrr": rr}
    for k in k_values:
        rels_k = rels[:k]
        ideal_k = ideal_rels[:k]
        hit = 1 if any(rels_k) else 0
        prec = sum(rels_k) / k
        # Recall denominator: any retrieved that match gold (approximation)
        total_rel = sum(rels)
        rec = (sum(rels_k) / total_rel) if total_rel else 0.0
        ndcg = (dcg(rels_k) / dcg(ideal_k)) if any(ideal_k) else 0.0
        row.update({
            f"hit@{k}": hit, f"prec@{k}": round(prec, 4),
            f"rec@{k}": round(rec, 4), f"ndcg@{k}": round(ndcg, 4),
        })
    return row

per_query = []
for i, row in df.iterrows():
    r = evaluate_query(row["query"], row.get("context", ""),
                       CONFIG["top_ks"], CONFIG["match_overlap"])
    per_query.append(r)
    print(f"[{i+1:>2}/{len(df)}]  MRR={r['mrr']:.3f}  hit@5={r['hit@5']}  "
          f"P@5={r['prec@5']:.2f}  nDCG@5={r['ndcg@5']:.2f}  ({r['latency_ms']:.0f}ms)")


## Cell 4 - Aggregate report + per-category breakdown

In [ ]:
dfm = pd.DataFrame(per_query)
ks = CONFIG["top_ks"]

agg = {"n_queries": len(dfm), "mean_mrr": round(dfm["mrr"].mean(), 4),
       "mean_latency_ms": round(dfm["latency_ms"].mean(), 1)}
for k in ks:
    agg[f"hit_rate@{k}"]    = round(dfm[f"hit@{k}"].mean(), 4)
    agg[f"mean_prec@{k}"]   = round(dfm[f"prec@{k}"].mean(), 4)
    agg[f"mean_recall@{k}"] = round(dfm[f"rec@{k}"].mean(), 4)
    agg[f"mean_ndcg@{k}"]   = round(dfm[f"ndcg@{k}"].mean(), 4)

report = {
    "embedding_model": CONFIG["embedding_model"],
    "match_overlap_threshold": CONFIG["match_overlap"],
    "aggregate": agg,
    "evaluated_at": datetime.now().isoformat(timespec="seconds"),
}
out = EVAL_DIR / f"retrieval_metrics_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
out.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(report, indent=2, ensure_ascii=False))
print(f"\nSaved: {out}")

detail_csv = EVAL_DIR / "retrieval_metrics_detail.csv"
dfm.to_csv(detail_csv, index=False, encoding="utf-8")
print(f"Per-query CSV: {detail_csv}")


## Cell 5 - Visualization (optional)

In [ ]:
try:
    import matplotlib.pyplot as plt
    ks = CONFIG["top_ks"]
    fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
    for ax, metric, title in zip(
        axes,
        ["hit", "prec", "rec", "ndcg"],
        ["Hit Rate @ k", "Precision @ k", "Recall @ k", "nDCG @ k"]
    ):
        vals = [dfm[f"{metric}@{k}"].mean() for k in ks]
        ax.plot(ks, vals, marker="o", lw=2)
        ax.set_title(title); ax.set_xlabel("k"); ax.set_xticks(ks)
        ax.set_ylim(0, 1.05); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(EVAL_DIR / "retrieval_metrics.png", dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Plot saved: {EVAL_DIR / 'retrieval_metrics.png'}")
except ImportError:
    print("matplotlib not installed; skip plotting.  pip install matplotlib")


---
### How to defend these numbers

- **MRR ~1.0** + **Hit Rate@1 ~1.0** = your retrieval is excellent; the right chunk is at the top.
- **Precision drops with k** is normal (you keep adding less-relevant chunks). Report P@1 and P@5.
- **Recall grows with k**. The interesting cutoff is where it plateaus (your sweet spot for `top_k`).
- **nDCG** rewards putting relevant chunks higher; a high MRR with a low nDCG would suggest mid-rank noise.

For your thesis, combine the RAGAS table (generation quality) and this table (retrieval quality)
side-by-side. That's the standard "two-level evaluation" pattern used in modern RAG papers.
